# Conceptos Básicos para Integrar Múltiples Ontologías

Cuando integras ontologías de diferentes fuentes, necesitas:
1. Mapear conceptos equivalentes entre ontologías
2. Establecer puentes entre vocabularios diferentes
3. Mantener la coherencia semántica
4. Reutilizar propiedades estándar cuando sea posible

### Importar Librerías

In [10]:
from rdflib import Graph, Namespace, URIRef, Literal, RDF, RDFS, OWL, XSD
import sbc_tools as sbc

# EQUIVALENCIA DE CLASES con OWL.equivalentClass


Cuando dos clases de diferentes ontologías representan el mismo concepto

Primero declaramos los namespaces de las diferentes ontologías a integrar

In [11]:
graph = Graph()

LOCAL = Namespace("http://example.org/music/")
MO = Namespace("http://purl.org/ontology/mo/")
SCHEMA = Namespace("https://schema.org/")

graph.bind("local", LOCAL)
graph.bind("mo", MO)
graph.bind("schema", SCHEMA)

Luego declaramos nuestra clase local ``Artista``

In [12]:
graph.add((LOCAL.Artista, RDF.type, OWL.Class))
graph.add((LOCAL.Artista, RDFS.label, Literal("Artista Musical")))

<Graph identifier=Na24cea0a1fe5452ab5a4c8fbfc10aa72 (<class 'rdflib.graph.Graph'>)>

### Declarar que la clase local es equivalente a clases de otras ontologías

**Estructura:**
```
Mi ontología:     local:Artista
                     ↓ owl:equivalentClass
Music Ontology:   mo:MusicArtist
                     ↓ owl:equivalentClass
Schema.org:       schema:MusicGroup
```

**SIGNIFICADO:** Estas tres clases representan el MISMO concepto. Un razonador OWL puede inferir que cualquier instancia de una también es instancia de las otras.

In [13]:
graph.add((LOCAL.Artista, OWL.equivalentClass, MO.MusicArtist))
graph.add((LOCAL.Artista, OWL.equivalentClass, SCHEMA.MusicGroup))

<Graph identifier=Na24cea0a1fe5452ab5a4c8fbfc10aa72 (<class 'rdflib.graph.Graph'>)>

### Crear instancia

**Con owl:equivalentClass, Bowie automáticamente es:**
- local:Artista (declarado explícitamente)
- mo:MusicArtist (inferido)
- schema:MusicGroup (inferido)

In [14]:
graph.add((LOCAL.Raphael, RDF.type, LOCAL.Artista))
graph.add((LOCAL.Raphael, RDFS.label, Literal("Raphael")))

<Graph identifier=Na24cea0a1fe5452ab5a4c8fbfc10aa72 (<class 'rdflib.graph.Graph'>)>

# EQUIVALENCIA DE PROPIEDADES con OWL.equivalentProperty

Cuando diferentes ontologías usan propiedades distintas para la misma relación

In [15]:

graph = Graph()

LOCAL = Namespace("http://example.org/music/")
MO = Namespace("http://purl.org/ontology/mo/")
SCHEMA = Namespace("https://schema.org/")
DC = Namespace("http://purl.org/dc/terms/")

graph.bind("local", LOCAL)
graph.bind("mo", MO)
graph.bind("schema", SCHEMA)
graph.bind("dcterms", DC)

### Mi propiedad local

In [16]:
graph.add((LOCAL.creador, RDF.type, OWL.ObjectProperty))
graph.add((LOCAL.creador, RDFS.label, Literal("creador")))

<Graph identifier=N829d893d06b149069aae1bdb4f7a1cf8 (<class 'rdflib.graph.Graph'>)>

### Mapear a propiedades de otras ontologías

**Estructura:**
```
Mi ontología:     local:creador
                     ↓ owl:equivalentProperty
Music Ontology:   mo:composer
                     ↓ owl:equivalentProperty
Schema.org:       schema:byArtist
                     ↓ owl:equivalentProperty
Dublin Core:      dcterms:creator
```

**SIGNIFICADO:** Todas estas propiedades expresan la MISMA relación.

In [17]:
graph.add((LOCAL.creador, OWL.equivalentProperty, MO.composer))
graph.add((LOCAL.creador, OWL.equivalentProperty, SCHEMA.byArtist))
graph.add((LOCAL.creador, OWL.equivalentProperty, DC.creator))

<Graph identifier=N829d893d06b149069aae1bdb4f7a1cf8 (<class 'rdflib.graph.Graph'>)>

### Usar la propiedad

**Al usar local:creador, también estamos diciendo:**
- mo:composer
- schema:byArtist
- dcterms:creator

In [18]:
graph.add((LOCAL.MigranNoche, LOCAL.creador, LOCAL.Raphael))

<Graph identifier=N829d893d06b149069aae1bdb4f7a1cf8 (<class 'rdflib.graph.Graph'>)>

# Subclases con RDFS:SUBCLASSOF


In [19]:
graph = Graph()

LOCAL = Namespace("http://example.org/music/")
MO = Namespace("http://purl.org/ontology/mo/")

graph.bind("local", LOCAL)
graph.bind("mo", MO)


**GuitarraMetal es un TIPO DE Guitarra**
- → Toda GuitarraMetal es una Guitarra
- → NO toda Guitarra es una GuitarraMetal

In [20]:
graph.add((LOCAL.GuitarraMetal, RDFS.subClassOf, LOCAL.Guitarra))

<Graph identifier=N0f13210015ca476fab17846a092998eb (<class 'rdflib.graph.Graph'>)>

# REUTILIZAR VOCABULARIOS ESTÁNDAR

No reinventar la rueda - usar vocabularios conocidos

In [21]:
graph = Graph()

LOCAL = Namespace("http://example.org/music/")
FOAF = Namespace("http://xmlns.com/foaf/0.1/")
DC = Namespace("http://purl.org/dc/terms/")
SCHEMA = Namespace("https://schema.org/")

graph.bind("local", LOCAL)
graph.bind("foaf", FOAF)
graph.bind("dcterms", DC)
graph.bind("schema", SCHEMA)

### Crear artista usando múltiples vocabularios estándar

In [22]:
graph.add((LOCAL.Raphael, RDF.type, LOCAL.Artista))

<Graph identifier=Ne5f09aaef70c4a7fa716ae4318f9b21d (<class 'rdflib.graph.Graph'>)>

### FOAF (Friend of a Friend) - para información sobre personas

In [23]:
graph.add((LOCAL.Raphael, FOAF.name, Literal("Raphael")))
graph.add((LOCAL.Raphael, FOAF.givenName, Literal("Raphael")))
graph.add((LOCAL.Raphael, FOAF.familyName, Literal("Martos Sánchez")))

<Graph identifier=Ne5f09aaef70c4a7fa716ae4318f9b21d (<class 'rdflib.graph.Graph'>)>

### Dublin Core - para metadatos

In [24]:
graph.add((LOCAL.Raphael, DC.description, Literal("Músico español atemporal")))

<Graph identifier=Ne5f09aaef70c4a7fa716ae4318f9b21d (<class 'rdflib.graph.Graph'>)>

### Schema.org - para datos estructurados web

In [25]:
graph.add((LOCAL.Raphael, SCHEMA.birthDate, Literal("1943-05-05")))
graph.add((LOCAL.Raphael, SCHEMA.nationality, Literal("Spanish")))

<Graph identifier=Ne5f09aaef70c4a7fa716ae4318f9b21d (<class 'rdflib.graph.Graph'>)>

## VOCABULARIOS ESTÁNDAR REUTILIZADOS:

### FOAF (Friend of a Friend):
→ foaf:name, foaf:givenName, foaf:familyName  
→ Estándar para describir personas

### Dublin Core (DC):
→ dcterms:creator, dcterms:title, dcterms:description  
→ Estándar para metadatos

### Schema.org:
→ schema:birthDate, schema:nationality, schema:url  
→ Usado por Google, Bing, Yahoo para datos estructurados

#  EJEMPLO COMPLETO DE INTEGRACIÓN

Cómo estructurar una ontología que integra múltiples fuentes

In [26]:
graph = Graph()

LOCAL = Namespace("http://example.org/music/")
WD = Namespace("http://www.wikidata.org/entity/")
MO = Namespace("http://purl.org/ontology/mo/")
SCHEMA = Namespace("https://schema.org/")
FOAF = Namespace("http://xmlns.com/foaf/0.1/")

graph.bind("local", LOCAL)
graph.bind("wd", WD)
graph.bind("mo", MO)
graph.bind("schema", SCHEMA)
graph.bind("foaf", FOAF)
graph.bind("owl", OWL)
graph.bind("rdfs", RDFS)

## PASO 1: Declarar la ontología

In [27]:
graph.add((LOCAL[""], RDF.type, OWL.Ontology))
graph.add((LOCAL[""], RDFS.label, Literal("Mi Ontología Musical Integrada")))

<Graph identifier=N0509e0f6cd8746868d86d7c7f43df3a9 (<class 'rdflib.graph.Graph'>)>

## PASO 2: Declarar importaciones de ontologías externas

In [28]:
graph.add((LOCAL[""], OWL.imports, URIRef("http://purl.org/ontology/mo/")))
graph.add((LOCAL[""], OWL.imports, URIRef("https://schema.org/")))

<Graph identifier=N0509e0f6cd8746868d86d7c7f43df3a9 (<class 'rdflib.graph.Graph'>)>

## PASO 3: Crear clases locales

In [29]:
graph.add((LOCAL.ArtistaMusical, RDF.type, OWL.Class))
graph.add((LOCAL.ArtistaMusical, RDFS.label, Literal("Artista Musical")))

<Graph identifier=N0509e0f6cd8746868d86d7c7f43df3a9 (<class 'rdflib.graph.Graph'>)>

## PASO 4: Mapear a ontologías externas

In [30]:
graph.add((LOCAL.ArtistaMusical, OWL.equivalentClass, MO.MusicArtist))
graph.add((LOCAL.ArtistaMusical, OWL.equivalentClass, SCHEMA.MusicGroup))

<Graph identifier=N0509e0f6cd8746868d86d7c7f43df3a9 (<class 'rdflib.graph.Graph'>)>

## PASO 5: Crear instancia con datos de Wikidata

Raphael en Wikidata

In [31]:
raphael_wd = WD.Q561205

### Declarar tipos (local + externas)

In [32]:
graph.add((raphael_wd, RDF.type, LOCAL.ArtistaMusical))
graph.add((raphael_wd, RDF.type, MO.MusicArtist))
graph.add((raphael_wd, RDF.type, SCHEMA.Person))

<Graph identifier=N0509e0f6cd8746868d86d7c7f43df3a9 (<class 'rdflib.graph.Graph'>)>

### Usar vocabularios estándar para propiedades

In [33]:
graph.add((raphael_wd, FOAF.name, Literal("Raphael")))
graph.add((raphael_wd, SCHEMA.birthDate, Literal("1943-05-05")))

<Graph identifier=N0509e0f6cd8746868d86d7c7f43df3a9 (<class 'rdflib.graph.Graph'>)>

### Propiedades personalizadas

In [34]:
graph.add((LOCAL.generoMusical, RDF.type, OWL.ObjectProperty))
graph.add((LOCAL.generoMusical, RDFS.label, Literal("género musical")))
graph.add((LOCAL.generoMusical, OWL.equivalentProperty, MO.genre))
graph.add((LOCAL.generoMusical, OWL.equivalentProperty, SCHEMA.genre))
graph.add((LOCAL.generoMusical, OWL.equivalentProperty, FOAF.topic))
graph.add((LOCAL.generoMusical, OWL.equivalentProperty, WD.P136))

graph.add((raphael_wd, LOCAL.generoMusical, LOCAL.Pop))
graph.add((raphael_wd, LOCAL.generoMusical, LOCAL.Glam))

<Graph identifier=N0509e0f6cd8746868d86d7c7f43df3a9 (<class 'rdflib.graph.Graph'>)>

### Lo visualizamos

In [35]:
sbc.show_graph(graph, output_file="ontology_musical_initial.html", select_menu=False, filter_menu=False)

out/ontology_musical_initial.html


Y guardamos

In [36]:
sbc.save(graph, "ontology_musical_initial.ttl")

Ontología guardada en: data\ontology_musical_initial.ttl


Y guardamos

## Ahora, podemos, incluso hacer una consulta SPARQL a nuestro propio grafo

In [37]:
query1 = f"""
PREFIX local: <{LOCAL}>

SELECT ?artist ?artistName WHERE {{
    ?artist a local:ArtistaMusical;
            foaf:name ?artistName.
}}
"""

print("Artistas")
results = graph.query(query1)
for row in results:
    artist = row.artistName
    print(artist)

Artistas
Raphael


# Ejercicios

## Añade otros artistas manualmente al grafo.
- Pedro Antonio Martín Peláez
- Álvaro ray Martínez

In [38]:
# añadimos mas artistas al grafo
walton_wd = WD.Q114510811

# añadimos a Rosa Walton
graph.add((walton_wd, RDF.type, LOCAL.ArtistaMusical))
graph.add((walton_wd, RDF.type, MO.MusicArtist))
graph.add((walton_wd, RDF.type, SCHEMA.Person))

graph.add((walton_wd, FOAF.name, Literal("Rosa Walton")))
graph.add((walton_wd, SCHEMA.birthDate, Literal("1994-10-28")))
graph.add((walton_wd, LOCAL.generoMusical, LOCAL.Pop))

# añadimos a "Dawid Podsiadlo"
davidP_wd = WD.Q5242353
graph.add((davidP_wd, RDF.type, LOCAL.ArtistaMusical))
graph.add((davidP_wd, RDF.type, MO.MusicArtist))
graph.add((davidP_wd, RDF.type, SCHEMA.Person))

graph.add((davidP_wd, FOAF.name, Literal("Dawid Podsiadlo")))
graph.add((davidP_wd, SCHEMA.birthDate, Literal("1993-05-25")))
graph.add((davidP_wd, LOCAL.generoMusical, LOCAL.Pop))




<Graph identifier=N0509e0f6cd8746868d86d7c7f43df3a9 (<class 'rdflib.graph.Graph'>)>

## Incluye alguna propiedad adicional (nacionalidad, premios, álbumes, ...). Declara la equivalencia de esas propiedades con otras ontologías.

In [39]:
# incluimos la nacionalidad
graph.add((LOCAL.nacionalidad, RDF.type, OWL.DatatypeProperty))
graph.add((LOCAL.nacionalidad, RDFS.label, Literal("nacionalidad")))
graph.add((LOCAL.nacionalidad, OWL.equivalentProperty, SCHEMA.nationality))
graph.add((LOCAL.nacionalidad, OWL.equivalentProperty, MO.nationality))
graph.add((LOCAL.nacionalidad, OWL.equivalentProperty, WD.P27))


graph.add((walton_wd, LOCAL.nacionalidad, Literal("Britanica")))
graph.add((raphael_wd, LOCAL.nacionalidad, Literal("Española")))
graph.add((davidP_wd, LOCAL.nacionalidad, Literal("Polaca")))

# incluimos los instrumentos que tocan
graph.add((LOCAL.instrumenToque, RDF.type, OWL.DatatypeProperty))
graph.add((LOCAL.instrumenToque, RDFS.label, Literal("instrumento que toca")))
graph.add((LOCAL.instrumenToque, OWL.equivalentProperty, MO.instrument))
graph.add((LOCAL.instrumenToque, OWL.equivalentProperty, SCHEMA.instrument))
graph.add((LOCAL.instrumenToque, OWL.equivalentProperty, WD.P1303))

graph.add((walton_wd, LOCAL.instrumenToque, Literal("Voz, Guitarra")))
graph.add((raphael_wd, LOCAL.instrumenToque, Literal("Voz")))
graph.add((davidP_wd, LOCAL.instrumenToque, Literal("Voz, Piano")))

<Graph identifier=N0509e0f6cd8746868d86d7c7f43df3a9 (<class 'rdflib.graph.Graph'>)>

## Crea una consulta SPARQL que utilice las propiedades adicionales.

In [40]:
# consulta con las nuevas propiedades
query2 = f"""
PREFIX local: <{LOCAL}>

SELECT ?artist ?artistName ?nationality ?instruments 
WHERE {{
    ?artista a local:ArtistaMusical;
            foaf:name ?artistName;
            local:nacionalidad ?nationality;
            local:instrumenToque ?instruments.

}}

"""

results = graph.query(query2)
print("\nArtistas con nacionalidad e instrumentos que tocan")
for row in results:
    artist = row.artistName
    nationality = row.nationality
    instruments = row.instruments

    print(f"Artista: {artist}, Nacionalidad: {nationality}, Instrumentos: {instruments}")

sbc.save(graph, "ontology_musical_integrated.ttl")


Artistas con nacionalidad e instrumentos que tocan
Artista: Raphael, Nacionalidad: Española, Instrumentos: Voz
Artista: Rosa Walton, Nacionalidad: Britanica, Instrumentos: Voz, Guitarra
Artista: Dawid Podsiadlo, Nacionalidad: Polaca, Instrumentos: Voz, Piano
Ontología guardada en: data\ontology_musical_integrated.ttl


In [42]:
sbc.show_graph(graph, output_file="ontology_musical_integrated.html")

out/ontology_musical_integrated.html
